# Data Analysis of Incarceration & Disenfranchisement

**Tools:** Python, Pandas, Matplotlib  
**Focus:** Racial disparities in incarceration and voting access (2005–2025)

## Project Overview
This notebook explores trends in incarceration and disenfranchisement data from 2005 to 2025. The goal is to identify patterns, visualize disparities, and connect quantitative findings to broader social issues.

## Research Questions
- How have incarceration trends changed over time?
- Are there visible racial disparities in incarceration rates?
- How does disenfranchisement vary across groups or years?
- What broader patterns emerge when comparing incarceration and voting access?

> **Note:** This notebook is designed to be portfolio-ready. If your column names differ, update the column mapping section below.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

## 1. Load Data

Place your dataset in the same GitHub repo and update the file path below if needed.

**Recommended repo structure:**
```text
incarceration-data-analysis/
├── data/
│   └── incarceration_data.csv
├── notebooks/
│   └── incarceration_analysis.ipynb
├── outputs/
│   └── charts/
└── README.md
```

In [ ]:
data_path = Path('../data/incarceration_data.csv')

if data_path.exists():
    df = pd.read_csv(data_path)
else:
    raise FileNotFoundError(
        f'Could not find the dataset at {data_path}. '\
        'Upload your CSV to data/incarceration_data.csv or update the path.'
    )

df.head()

## 2. Inspect the Dataset

Use this section to understand the structure of the data before cleaning.

In [ ]:
print('Shape:', df.shape)
print('\nColumns:')
print(df.columns.tolist())
print('\nMissing values:')
print(df.isna().sum())

## 3. Clean and Standardize Data

This section standardizes column names and prepares the data for analysis.

In [ ]:
# Standardize column names
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('-', '_')
)

df.head()

### Column Mapping

Update the dictionary below so it matches your actual dataset.  
This lets the rest of the notebook stay clean and readable.

In [ ]:
# EDIT THESE VALUES TO MATCH YOUR CSV COLUMNS
column_map = {
    'year': 'year',
    'race': 'race',
    'incarceration_rate': 'incarceration_rate',
    'disenfranchised_population': 'disenfranchised_population',
    'state': 'state',  # optional
}

# Keep only mapped columns that actually exist in the dataset
available_map = {k: v for k, v in column_map.items() if v in df.columns}
analysis_df = df.rename(columns={v: k for k, v in available_map.items()}).copy()

analysis_df.head()

In [ ]:
# Basic type cleanup
if 'year' in analysis_df.columns:
    analysis_df['year'] = pd.to_numeric(analysis_df['year'], errors='coerce')

for col in ['incarceration_rate', 'disenfranchised_population']:
    if col in analysis_df.columns:
        analysis_df[col] = pd.to_numeric(analysis_df[col], errors='coerce')

analysis_df = analysis_df.dropna(subset=[c for c in ['year'] if c in analysis_df.columns])
analysis_df.head()

## 4. Exploratory Data Analysis

This section summarizes key features of the dataset.

In [ ]:
analysis_df.describe(include='all')

In [ ]:
if 'race' in analysis_df.columns:
    analysis_df['race'].value_counts(dropna=False)

## 5. Visualization 1: Incarceration Trends Over Time

In [ ]:
output_dir = Path('../outputs/charts')
output_dir.mkdir(parents=True, exist_ok=True)

if {'year', 'incarceration_rate'}.issubset(analysis_df.columns):
    yearly_trend = analysis_df.groupby('year', as_index=False)['incarceration_rate'].mean()

    plt.figure(figsize=(10, 6))
    plt.plot(yearly_trend['year'], yearly_trend['incarceration_rate'])
    plt.title('Average Incarceration Rate Over Time')
    plt.xlabel('Year')
    plt.ylabel('Incarceration Rate')
    plt.tight_layout()
    plt.savefig(output_dir / 'incarceration_trend.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('This chart requires year and incarceration_rate columns.')

## 6. Visualization 2: Average Incarceration Rate by Race

In [ ]:
if {'race', 'incarceration_rate'}.issubset(analysis_df.columns):
    racial_rates = (
        analysis_df.groupby('race', as_index=False)['incarceration_rate']
        .mean()
        .sort_values('incarceration_rate', ascending=False)
    )

    plt.figure(figsize=(10, 6))
    plt.bar(racial_rates['race'], racial_rates['incarceration_rate'])
    plt.title('Average Incarceration Rate by Race')
    plt.xlabel('Race')
    plt.ylabel('Average Incarceration Rate')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(output_dir / 'racial_disparities.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('This chart requires race and incarceration_rate columns.')

## 7. Visualization 3: Disenfranchisement Over Time

In [ ]:
if {'year', 'disenfranchised_population'}.issubset(analysis_df.columns):
    disenfranchisement_trend = analysis_df.groupby('year', as_index=False)['disenfranchised_population'].sum()

    plt.figure(figsize=(10, 6))
    plt.plot(disenfranchisement_trend['year'], disenfranchisement_trend['disenfranchised_population'])
    plt.title('Disenfranchised Population Over Time')
    plt.xlabel('Year')
    plt.ylabel('Disenfranchised Population')
    plt.tight_layout()
    plt.savefig(output_dir / 'voting_access.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print('This chart requires year and disenfranchised_population columns.')

## 8. Optional Comparison: Race and Disenfranchisement

In [ ]:
if {'race', 'disenfranchised_population'}.issubset(analysis_df.columns):
    race_disenfranchisement = (
        analysis_df.groupby('race', as_index=False)['disenfranchised_population']
        .sum()
        .sort_values('disenfranchised_population', ascending=False)
    )
    display(race_disenfranchisement)
else:
    print('This table requires race and disenfranchised_population columns.')

## 9. Findings

Update this section after you run the notebook and review your charts.

**Example write-up:**
- Incarceration trends changed over time, but disparities remained visible across groups.
- Racial differences in incarceration rates suggest unequal outcomes in the criminal justice system.
- Disenfranchisement data highlights how incarceration can affect civic participation and voting access.
- Together, these patterns suggest broader structural inequalities rather than isolated differences.

## 10. Conclusion

This project uses data analysis and visualization to examine the intersection of incarceration and disenfranchisement. Beyond describing trends, the analysis helps connect numerical patterns to broader questions about racial inequality, access to voting, and long-term social impact.

For a stronger portfolio project, make sure your final GitHub repo includes:
- this notebook,
- your dataset,
- saved chart images,
- and a polished README.